# SPARQL-to-Cypher Transpiler — End-to-End Demo

This notebook demonstrates `rdflib_neo4j.sparql.transpiler.translate` — a function
that converts SPARQL SELECT queries to Cypher 25 by walking the rdflib algebra AST.

We use a small **FOAF social network** loaded via rdflib-neo4j:
four persons (Alice, Bob, Carol, Dave) with names, ages, departments, and social links.

The notebook covers:
1. Loading the dataset into Neo4j via rdflib-neo4j
2. Basic graph patterns (BGP)
3. Filters
4. Joins across patterns
5. OPTIONAL (LeftJoin)
6. UNION
7. BIND expressions
8. GROUP BY + aggregates
9. MINUS
10. VALUES
11. Two-hop relationship chains
12. Full combined query
13. Advanced queries (HAVING, friend-of-friend, BIND with defaults, MINUS relationship)
14. Inspecting the rdflib algebra
15. Interactive cell

## 0. Setup

In [1]:
%pip install -q rdflib-neo4j neo4j python-dotenv

/Users/mh/d/python/rdflib-neo4j/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()  # reads NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, NEO4J_DATABASE from .env

URI      = os.environ["NEO4J_URI"]
USER     = os.environ["NEO4J_USERNAME"]
PASSWORD = os.environ["NEO4J_PASSWORD"]
DATABASE = os.environ.get("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print(f"Connected to {URI} / {DATABASE}")

Connected to bolt://localhost / neo4j


## 1. Load Sample Data

In [3]:
TURTLE = """
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix ex:   <http://example.org/> .

ex:alice a foaf:Person ;
    foaf:name  "Alice" ;
    foaf:age   30 ;
    foaf:mbox  "alice@example.org" ;
    foaf:knows ex:bob .

ex:bob a foaf:Person ;
    foaf:name  "Bob" ;
    foaf:age   25 ;
    foaf:mbox  "bob@example.org" ;
    foaf:member "engineering" .

ex:carol a foaf:Person ;
    foaf:name   "Carol" ;
    foaf:age    35 ;
    foaf:member "engineering" ;
    foaf:knows  ex:alice .

ex:dave a foaf:Person ;
    foaf:name   "Dave" ;
    foaf:age    28 ;
    foaf:member "product" .
"""

from rdflib import Graph
from rdflib_neo4j import Neo4jStoreConfig, Neo4jStore, HANDLE_VOCAB_URI_STRATEGY

# Clear + ensure constraint
driver.execute_query("MATCH (n:Resource) DETACH DELETE n", database_=DATABASE)
driver.execute_query(
    "CREATE CONSTRAINT n10s_unique_uri IF NOT EXISTS "
    "FOR (r:Resource) REQUIRE r.uri IS UNIQUE",
    database_=DATABASE,
)

auth_data = {"uri": URI, "database": DATABASE, "user": USER, "pwd": PASSWORD}
config = Neo4jStoreConfig(
    auth_data=auth_data,
    custom_prefixes={"foaf": "http://xmlns.com/foaf/0.1/"},
    handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
    batching=False,
)
g = Graph(store=Neo4jStore(config=config))
g.open(config, create=True)
g.parse(data=TURTLE, format="ttl")
g.close(True)

count, _, _ = driver.execute_query("MATCH (n:Person) RETURN count(n) AS c", database_=DATABASE)
print(f"Loaded {count[0]['c']} Person nodes")

Uniqueness constraint on :Resource(uri) is created.
The store is now: Open
Uniqueness constraint on :Resource(uri) is created.
The store is now: Open
The store is now: Closed
IMPORTED 19 TRIPLES
Loaded 4 Person nodes


### Helper: translate + run

In [4]:
from rdflib_neo4j.sparql.transpiler import translate

FOAF = "http://xmlns.com/foaf/0.1/"
EX   = "http://example.org/"

def run_sparql(sparql: str, show_cypher: bool = True):
    cypher, params = translate(sparql, config)
    if show_cypher:
        print("── Cypher " + "─" * 50)
        print(cypher)
        if params:
            print("params:", params)
        print("─" * 59)
    records, _, _ = driver.execute_query(cypher, params, database_=DATABASE)
    return records

## 2. Basic Graph Pattern (BGP)

The transpiler recognises three BGP triple forms:
- `?p a :Person` → `MATCH (p:Person)`  
- `?p :name ?name` → property binding `p.\`name\` AS name`  
- `?p :name "Alice"` → `WHERE p.\`name\` = "Alice"`

In [5]:
sparql = f"""
SELECT ?name ?age WHERE {{
  ?p a <{FOAF}Person> ;
     <{FOAF}name> ?name ;
     <{FOAF}age>  ?age .
}}
"""

for r in run_sparql(sparql):
    print(r["name"], r["age"])

── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Person)
WHERE p.`age` IS NOT NULL AND p.`name` IS NOT NULL
RETURN p.`name` AS name, p.`age` AS age
───────────────────────────────────────────────────────────
Alice 30
Bob 25
Carol 35
Dave 28


## 3. FILTER

SPARQL FILTER maps to Cypher WHERE. The transpiler handles:
- Comparison operators (`>`, `<`, `>=`, `<=`, `=`, `!=`)
- `&&` / `||` → `AND` / `OR`
- `BOUND(?x)` → `IS NOT NULL`
- `REGEX(?x, pattern)` → `=~`
- String functions: `UCASE`, `LCASE`, `STRLEN`, `STRSTARTS`, `CONTAINS`

In [6]:
sparql = f"""
SELECT ?name ?age WHERE {{
  ?p a <{FOAF}Person> ;
     <{FOAF}name> ?name ;
     <{FOAF}age>  ?age .
  FILTER(?age >= 28 && BOUND(?name))
}} ORDER BY ?age
"""

for r in run_sparql(sparql):
    print(r["name"], r["age"])

── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Person)
WHERE p.`age` IS NOT NULL AND p.`name` IS NOT NULL AND p.`age` >= 28 AND p.`name` IS NOT NULL
RETURN p.`name` AS name, p.`age` AS age
ORDER BY p.`age` ASC
───────────────────────────────────────────────────────────
Dave 28
Alice 30
Carol 35


In [7]:
# REGEX filter
sparql = f"""
SELECT ?name WHERE {{
  ?p a <{FOAF}Person> ; <{FOAF}name> ?name .
  FILTER(REGEX(?name, "^[AB]"))
}}
"""

for r in run_sparql(sparql):
    print(r["name"])

── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Person)
WHERE p.`name` IS NOT NULL AND p.`name` =~ "^[AB].*"
RETURN p.`name` AS name
───────────────────────────────────────────────────────────
Alice
Bob


## 4. Join — Two BGPs Sharing a Variable

When two BGP patterns share a variable (e.g. `?p`), the transpiler translates both
patterns into the same `CypherQuery` — they automatically join on the shared node.

In [8]:
sparql = f"""
SELECT ?name ?dept WHERE {{
  ?p a <{FOAF}Person> ; <{FOAF}name> ?name .
  ?p <{FOAF}member> ?dept .
}}
"""

for r in run_sparql(sparql):
    print(r["name"], "→", r["dept"])

── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Person)
WHERE p.`member` IS NOT NULL AND p.`name` IS NOT NULL
RETURN p.`name` AS name, p.`member` AS dept
───────────────────────────────────────────────────────────
Bob → engineering
Carol → engineering
Dave → product


## 5. Relationship Pattern (two-hop)

When a variable appears as a **subject** elsewhere in the query, the transpiler
classifies it as a graph node and emits `MATCH (p)-[:knows]->(f)` instead of
a property access. This is the *subject-variable heuristic*.

In [9]:
# Alice knows Bob; what is Bob's name?
sparql = f"""
SELECT ?pname ?fname WHERE {{
  ?p a <{FOAF}Person> ; <{FOAF}name> ?pname .
  ?p <{FOAF}knows> ?f .
  ?f <{FOAF}name> ?fname .
}}
"""

for r in run_sparql(sparql):
    print(r["pname"], "→ knows →", r["fname"])

── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Person)
MATCH (f:Resource)
MATCH (p)-[:knows]->(f)
WHERE p.`name` IS NOT NULL AND f.`name` IS NOT NULL
RETURN p.`name` AS pname, f.`name` AS fname
───────────────────────────────────────────────────────────
Alice → knows → Bob
Carol → knows → Alice


## 6. OPTIONAL (LeftJoin)

SPARQL `OPTIONAL { … }` → Cypher `OPTIONAL MATCH` when the optional variable
is a subject-variable (node), otherwise the property may just be `null`.

In [10]:
sparql = f"""
SELECT ?name ?email ?fname WHERE {{
  ?p a <{FOAF}Person> ; <{FOAF}name> ?name .
  OPTIONAL {{ ?p <{FOAF}mbox> ?email }}
  OPTIONAL {{
    ?p <{FOAF}knows> ?f .
    ?f <{FOAF}name> ?fname .
  }}
}} ORDER BY ?name
"""

for r in run_sparql(sparql):
    print(f"{r['name']:8}  email={r['email']}  knows={r['fname']}")

── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Person)
WHERE p.`name` IS NOT NULL
OPTIONAL MATCH (p)-[:knows]->(f:Resource)
RETURN p.`name` AS name, p.`mbox` AS email, f.`name` AS fname
ORDER BY p.`name` ASC
───────────────────────────────────────────────────────────
Alice     email=alice@example.org  knows=Bob
Bob       email=bob@example.org  knows=None
Carol     email=None  knows=Alice
Dave      email=None  knows=None


## 7. UNION

SPARQL UNION → Cypher UNION. Each branch produces its own sub-query.

In [11]:
sparql = f"""
SELECT ?name WHERE {{
  {{ ?p <{FOAF}name> ?name . FILTER(?name = "Alice") }}
  UNION
  {{ ?p <{FOAF}name> ?name . FILTER(?name = "Dave") }}
}}
"""

for r in run_sparql(sparql):
    print(r["name"])

── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Resource)
WHERE p.`name` IS NOT NULL AND p.`name` = "Alice"
RETURN p.`name` AS name
UNION
MATCH (p:Resource)
WHERE p.`name` IS NOT NULL AND p.`name` = "Dave"
RETURN p.`name` AS name
───────────────────────────────────────────────────────────
Alice
Dave


## 8. BIND — Computed Columns

`BIND(expr AS ?var)` adds a computed column. The transpiler supports
`UCASE/LCASE`, `CONCAT`, `COALESCE`, `IF` (→ `CASE WHEN`), and arithmetic.

In [12]:
sparql = f"""
SELECT ?name ?upper ?label WHERE {{
  ?p a <{FOAF}Person> ; <{FOAF}name> ?name ; <{FOAF}age> ?age .
  BIND(UCASE(?name) AS ?upper)
  BIND(IF(?age >= 30, "senior", "junior") AS ?label)
}} ORDER BY ?name
"""

for r in run_sparql(sparql):
    print(f"{r['name']:8} → {r['upper']:8}  ({r['label']})")

── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Person)
WHERE p.`age` IS NOT NULL AND p.`name` IS NOT NULL
RETURN p.`name` AS name, toUpper(p.`name`) AS upper, CASE WHEN p.`age` >= 30 THEN "senior" ELSE "junior" END AS label
ORDER BY p.`name` ASC
───────────────────────────────────────────────────────────
Alice    → ALICE     (senior)
Bob      → BOB       (junior)
Carol    → CAROL     (senior)
Dave     → DAVE      (junior)


## 9. GROUP BY + Aggregates

SPARQL aggregates map to Cypher's `WITH … count()/avg()/…` grouping pattern.
`GROUP_CONCAT` maps to `reduce(acc, x IN collect(x) | …)`.

In [13]:
# Count and average age per department
sparql = f"""
SELECT ?dept (COUNT(?p) AS ?cnt) (AVG(?age) AS ?avgAge) WHERE {{
  ?p <{FOAF}member> ?dept ;
     <{FOAF}age> ?age .
}} GROUP BY ?dept ORDER BY ?dept
"""

for r in run_sparql(sparql):
    print(f"{r['dept']:15}  count={r['cnt']}  avg_age={r['avgAge']:.1f}")

── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Resource)
WHERE p.`age` IS NOT NULL AND p.`member` IS NOT NULL
WITH p.`member` AS dept, count(p) AS __agg_1__, avg(p.`age`) AS __agg_2__
RETURN dept, __agg_1__ AS cnt, __agg_2__ AS avgAge
ORDER BY dept ASC
───────────────────────────────────────────────────────────
engineering      count=2  avg_age=30.0
product          count=1  avg_age=28.0


In [14]:
# GROUP_CONCAT — list of names per department
sparql = f"""
SELECT ?dept (GROUP_CONCAT(?name; SEPARATOR=", ") AS ?names) WHERE {{
  ?p <{FOAF}member> ?dept ; <{FOAF}name> ?name .
}} GROUP BY ?dept ORDER BY ?dept
"""

for r in run_sparql(sparql):
    print(f"{r['dept']:15}  → {r['names']}")

── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Resource)
WHERE p.`member` IS NOT NULL AND p.`name` IS NOT NULL
WITH p.`member` AS dept, reduce(__s = "", __n IN collect(p.`name`) | CASE WHEN __s = "" THEN __n ELSE __s + ", " + __n END) AS __agg_1__
RETURN dept, __agg_1__ AS names
ORDER BY dept ASC
───────────────────────────────────────────────────────────
engineering      → Bob, Carol
product          → Dave


## 10. MINUS

SPARQL MINUS → Cypher either negated `WHERE` or `NOT EXISTS { … }` subquery predicate.

In [15]:
# Persons who are NOT in engineering
sparql = f"""
SELECT ?name WHERE {{
  ?p a <{FOAF}Person> ; <{FOAF}name> ?name .
  MINUS {{ ?p <{FOAF}member> "engineering" }}
}} ORDER BY ?name
"""

for r in run_sparql(sparql):
    print(r["name"])

── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Person)
WHERE p.`name` IS NOT NULL AND NOT coalesce(p.`member` = "engineering", false)
RETURN p.`name` AS name
ORDER BY p.`name` ASC
───────────────────────────────────────────────────────────
Alice
Dave


## 11. VALUES

SPARQL VALUES constrains variables to a fixed set of values.
- **Bound** (variable already in scope): → `IN` predicate
- **Unbound** (new variable): → `UNWIND` the list

In [16]:
# Bound VALUES — filter to known names
sparql = f"""
SELECT ?name ?age WHERE {{
  ?p a <{FOAF}Person> ; <{FOAF}name> ?name ; <{FOAF}age> ?age .
  VALUES ?name {{ "Alice" "Dave" }}
}} ORDER BY ?name
"""

for r in run_sparql(sparql):
    print(r["name"], r["age"])

── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Person)
WHERE p.`age` IS NOT NULL AND p.`name` IS NOT NULL AND p.`name` IN $p0
RETURN p.`name` AS name, p.`age` AS age
ORDER BY p.`name` ASC
params: {'p0': ['Alice', 'Dave']}
───────────────────────────────────────────────────────────
Alice 30
Dave 28


## 12. Full Combined Query

Everything together: type, multiple properties, filter, order, limit.

In [17]:
sparql = f"""
SELECT ?name ?age ?dept WHERE {{
  ?p a <{FOAF}Person> ;
     <{FOAF}name>   ?name ;
     <{FOAF}age>    ?age .
  OPTIONAL {{ ?p <{FOAF}member> ?dept }}
  FILTER(?age >= 25)
}} ORDER BY ?age DESC(?name) LIMIT 10
"""

for r in run_sparql(sparql):
    print(f"{r['name']:8}  age={r['age']}  dept={r['dept']}")

── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Person)
WHERE p.`age` IS NOT NULL AND p.`name` IS NOT NULL AND p.`age` >= 25
RETURN p.`name` AS name, p.`age` AS age, p.`member` AS dept
ORDER BY p.`age` ASC, p.`name` DESC
LIMIT 10
───────────────────────────────────────────────────────────
Bob       age=25  dept=engineering
Dave      age=28  dept=product
Alice     age=30  dept=None
Carol     age=35  dept=engineering


## 13. Advanced Queries

More complex SPARQL patterns showing HAVING, multi-hop chains, and BIND with defaults.

In [18]:
# HAVING — departments with more than one member
sparql = f"""
    SELECT ?dept (COUNT(?p) AS ?n) WHERE {{
      ?p <http://xmlns.com/foaf/0.1/member> ?dept .
    }} GROUP BY ?dept HAVING (COUNT(?p) > 1)
"""

rows = run_sparql(sparql)
for r in rows:
    print(f"{r['dept']}  members={r['n']}")

assert any(r["dept"] == "engineering" for r in run_sparql(sparql, show_cypher=False)), \
    "engineering should have >1 member"


── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Resource)
WHERE p.`member` IS NOT NULL
WITH p.`member` AS dept, count(p) AS __agg_1__, count(p) AS __agg_2__
WHERE __agg_2__ > 1
RETURN dept, __agg_1__ AS n
───────────────────────────────────────────────────────────
engineering  members=2


In [19]:
# Friend-of-friend: who do Alice's friends know?
sparql = f"""
    SELECT ?pname ?fname ?ffname WHERE {{
      ?p <http://xmlns.com/foaf/0.1/name> ?pname .
      ?p <http://xmlns.com/foaf/0.1/knows> ?f .
      ?f <http://xmlns.com/foaf/0.1/name> ?fname .
      ?f <http://xmlns.com/foaf/0.1/knows> ?ff .
      ?ff <http://xmlns.com/foaf/0.1/name> ?ffname .
      FILTER(?pname = "Alice")
    }} ORDER BY ?fname ?ffname
"""

for r in run_sparql(sparql):
    print(f"{r['pname']} → {r['fname']} → {r['ffname']}")


── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (f:Resource)
MATCH (ff:Resource)
MATCH (p:Resource)
MATCH (f)-[:knows]->(ff)
MATCH (p)-[:knows]->(f)
WHERE f.`name` IS NOT NULL AND ff.`name` IS NOT NULL AND p.`name` IS NOT NULL AND p.`name` = "Alice"
RETURN p.`name` AS pname, f.`name` AS fname, ff.`name` AS ffname
ORDER BY f.`name` ASC, ff.`name` ASC
───────────────────────────────────────────────────────────


In [20]:
# BIND with IF(BOUND()) — default department label for persons without a dept
sparql = f"""
    SELECT ?name ?deptLabel WHERE {{
      ?p a <http://xmlns.com/foaf/0.1/Person> ; <http://xmlns.com/foaf/0.1/name> ?name .
      OPTIONAL {{ ?p <http://xmlns.com/foaf/0.1/member> ?dept }}
      BIND(IF(BOUND(?dept), ?dept, "(none)") AS ?deptLabel)
    }} ORDER BY ?name
"""

for r in run_sparql(sparql):
    print(f"{r['name']:8}  dept={r['deptLabel']}")


── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Person)
WHERE p.`name` IS NOT NULL
RETURN p.`name` AS name, CASE WHEN p.`member` IS NOT NULL THEN p.`member` ELSE "(none)" END AS deptLabel
ORDER BY p.`name` ASC
───────────────────────────────────────────────────────────
Alice     dept=(none)
Bob       dept=engineering
Carol     dept=engineering
Dave      dept=product


In [21]:
# MINUS with relationship — persons who do NOT know anyone
sparql = f"""
    SELECT ?name WHERE {{
      ?p a <http://xmlns.com/foaf/0.1/Person> ; <http://xmlns.com/foaf/0.1/name> ?name .
      MINUS {{ ?p <http://xmlns.com/foaf/0.1/knows> ?anyone }}
    }} ORDER BY ?name
"""

rows = run_sparql(sparql)
for r in rows:
    print(r["name"])


── Cypher ──────────────────────────────────────────────────
CYPHER 25
MATCH (p:Person)
WHERE p.`name` IS NOT NULL AND NOT EXISTS { MATCH (anyone:Resource) MATCH (p)-[:knows]->(anyone) RETURN anyone AS anyone }
RETURN p.`name` AS name
ORDER BY p.`name` ASC
───────────────────────────────────────────────────────────
Bob
Dave


## 14. Inspecting the rdflib Algebra

The transpiler walks the algebra tree produced by `rdflib.plugins.sparql.prepareQuery`.
You can inspect the tree to understand how SPARQL maps to algebra nodes.

In [22]:
from rdflib.plugins.sparql import prepareQuery

sparql = f"""
SELECT ?name ?age WHERE {{
  ?p a <{FOAF}Person> ; <{FOAF}name> ?name ; <{FOAF}age> ?age .
  FILTER(?age >= 18)
}} ORDER BY ?name LIMIT 5
"""

q = prepareQuery(sparql)

def show_algebra(node, depth=0):
    from rdflib.plugins.sparql.parserutils import CompValue
    indent = "  " * depth
    if isinstance(node, CompValue):
        print(f"{indent}{node.name}")
        for k in node.keys():
            v = node[k]
            if isinstance(v, CompValue):
                print(f"{indent}  .{k}:")
                show_algebra(v, depth + 2)
            elif isinstance(v, list) and v and isinstance(v[0], CompValue):
                print(f"{indent}  .{k}: [{', '.join(n.name for n in v)}]")
            else:
                print(f"{indent}  .{k}: {v!r}")
    else:
        print(f"{indent}{node!r}")

show_algebra(q.algebra)

SelectQuery
  .p:
    Slice
      .p:
        Project
          .p:
            OrderBy
              .p:
                Filter
                  .expr:
                    RelationalExpression
                      .expr: rdflib.term.Variable('age')
                      .op: '>='
                      .other: rdflib.term.Literal('18', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))
                      ._vars: set()
                  .p:
                    BGP
                      .triples: [(rdflib.term.Variable('p'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://xmlns.com/foaf/0.1/Person')), (rdflib.term.Variable('p'), rdflib.term.URIRef('http://xmlns.com/foaf/0.1/age'), rdflib.term.Variable('age')), (rdflib.term.Variable('p'), rdflib.term.URIRef('http://xmlns.com/foaf/0.1/name'), rdflib.term.Variable('name'))]
                      ._vars: {rdflib.term.Variable('name'), rdflib.term.Variable('p'), rdfl

## 15. Interactive: Try Your Own SPARQL

Edit the SPARQL query in the text area below and click **Translate & Run**.
The cell shows the generated Cypher and the query results live.

_Requires `ipywidgets` — already included in Colab and standard Jupyter._

In [23]:
import ipywidgets as widgets
from IPython.display import display, clear_output

_DEFAULT_SPARQL = """PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX ex:   <http://example.org/>

SELECT ?name ?age WHERE {
  ?p a foaf:Person ;
     foaf:name ?name ;
     foaf:age  ?age .
  FILTER(?age >= 25)
}
ORDER BY ?age
"""

sparql_input = widgets.Textarea(
    value=_DEFAULT_SPARQL.strip(),
    placeholder="Enter SPARQL SELECT query…",
    layout=widgets.Layout(width="100%", height="200px"),
)
run_btn = widgets.Button(
    description="Translate & Run",
    button_style="primary",
    icon="play",
)
out = widgets.Output()

def _on_run(b):
    with out:
        clear_output(wait=True)
        try:
            rows = run_sparql(sparql_input.value)
            if rows:
                print(f"\n── Results ({len(rows)} rows) " + "─" * 40)
                for r in rows:
                    print(dict(r))
            else:
                print("\n(no results)")
        except Exception as exc:
            print(f"\nError: {exc}")

run_btn.on_click(_on_run)
display(sparql_input, run_btn, out)


In [24]:
# driver.close()